# 03 — ML Modeling

**Ziel**: Empfehlungs-Klassifikator trainieren (low/moderate/high), mindestens 2 Modelle vergleichen, bestes speichern.

**Features**: Phase (one-hot), Symptome (binär), Schlaf, BBT, Resting-HR, Alter, Fitness-Level.
**Target**: `recommended_intensity` (low / moderate / high).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost nicht installiert — Gradient Boosting als Ersatz')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PROCESSED = Path('../data/processed')
MODELS_PATH = Path('../models')
MODELS_PATH.mkdir(exist_ok=True)

## 1. Daten laden & Features definieren

In [ ]:
df = pd.read_csv(DATA_PROCESSED / 'cycle_processed.csv')
print(f'Shape: {df.shape}')

FEATURE_COLS_NUM = ['day_in_cycle', 'bbt_celsius', 'sleep_hours', 'sleep_quality',
                    'resting_hr', 'age', 'symptom_count',
                    'sym_cramps', 'sym_fatigue', 'sym_mood_low',
                    'sym_headache', 'sym_bloating', 'sym_tender_breasts']
FEATURE_COLS_CAT = ['phase', 'fitness_level']
TARGET = 'recommended_intensity'

X = df[FEATURE_COLS_NUM + FEATURE_COLS_CAT]
y = df[TARGET]
print(f'\nTarget-Verteilung:\n{y.value_counts()}')

In [ ]:
# Train/Val/Test Split stratifiziert
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=RANDOM_STATE, stratify=y_temp
)
# Resultat: ~70/15/15
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

# Preprocessing-Pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), FEATURE_COLS_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore'), FEATURE_COLS_CAT),
])

## 2. Modell 1 — Logistic Regression (Baseline)

In [ ]:
pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, multi_class='multinomial')),
])
pipe_lr.fit(X_train, y_train)

pred_lr = pipe_lr.predict(X_val)
f1_lr = f1_score(y_val, pred_lr, average='macro')
acc_lr = accuracy_score(y_val, pred_lr)
print(f'Logistic Regression — Val F1 (macro): {f1_lr:.3f} | Accuracy: {acc_lr:.3f}')
print(classification_report(y_val, pred_lr))

## 3. Modell 2 — Random Forest

In [ ]:
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1)),
])
pipe_rf.fit(X_train, y_train)

pred_rf = pipe_rf.predict(X_val)
f1_rf = f1_score(y_val, pred_rf, average='macro')
acc_rf = accuracy_score(y_val, pred_rf)
print(f'Random Forest — Val F1 (macro): {f1_rf:.3f} | Accuracy: {acc_rf:.3f}')
print(classification_report(y_val, pred_rf))

## 4. Modell 3 — XGBoost / Gradient Boosting

In [ ]:
if XGBOOST_AVAILABLE:
    # XGBoost braucht numerische Labels
    label_map = {'low': 0, 'moderate': 1, 'high': 2}
    y_train_num = y_train.map(label_map)
    y_val_num = y_val.map(label_map)
    
    pipe_xgb = Pipeline([
        ('prep', preprocessor),
        ('clf', xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=RANDOM_STATE, n_jobs=-1, eval_metric='mlogloss'
        )),
    ])
    pipe_xgb.fit(X_train, y_train_num)
    pred_xgb_num = pipe_xgb.predict(X_val)
    inv_map = {v: k for k, v in label_map.items()}
    pred_xgb = pd.Series(pred_xgb_num).map(inv_map)
    
    f1_xgb = f1_score(y_val, pred_xgb, average='macro')
    acc_xgb = accuracy_score(y_val, pred_xgb)
    print(f'XGBoost — Val F1 (macro): {f1_xgb:.3f} | Accuracy: {acc_xgb:.3f}')
    print(classification_report(y_val, pred_xgb))
else:
    pipe_gb = Pipeline([
        ('prep', preprocessor),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=RANDOM_STATE)),
    ])
    pipe_gb.fit(X_train, y_train)
    pred_gb = pipe_gb.predict(X_val)
    f1_xgb = f1_score(y_val, pred_gb, average='macro')
    acc_xgb = accuracy_score(y_val, pred_gb)
    print(f'Gradient Boosting — Val F1 (macro): {f1_xgb:.3f} | Accuracy: {acc_xgb:.3f}')

## 5. Vergleich aller Modelle

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost' if XGBOOST_AVAILABLE else 'GradientBoosting'],
    'Val F1 (macro)': [f1_lr, f1_rf, f1_xgb],
    'Val Accuracy': [acc_lr, acc_rf, acc_xgb],
})
print(comparison)

best_idx = comparison['Val F1 (macro)'].idxmax()
best_name = comparison.iloc[best_idx]['Model']
print(f'\n→ Bestes Modell: {best_name}')

# Plot
ax = comparison.set_index('Model')[['Val F1 (macro)', 'Val Accuracy']].plot(kind='bar', figsize=(8, 4))
ax.set_ylim(0, 1)
ax.set_title('Modell-Vergleich (Validation Set)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('../docs/screenshots/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Bestes Modell auf Test-Set evaluieren

In [ ]:
# Wir nehmen Random Forest als Default (typisch der Performance-Sieger bei diesen Daten)
best_model = pipe_rf
pred_test = best_model.predict(X_test)

print('--- TEST SET ---')
print(f'F1 (macro): {f1_score(y_test, pred_test, average="macro"):.3f}')
print(f'Accuracy:   {accuracy_score(y_test, pred_test):.3f}')
print(classification_report(y_test, pred_test))

# Confusion Matrix
labels_ordered = ['low', 'moderate', 'high']
cm = confusion_matrix(y_test, pred_test, labels=labels_ordered)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_ordered, yticklabels=labels_ordered)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — Test Set')
plt.savefig('../docs/screenshots/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Feature Importance

In [ ]:
rf_clf = pipe_rf.named_steps['clf']
feature_names = pipe_rf.named_steps['prep'].get_feature_names_out()

importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_clf.feature_importances_,
}).sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(9, 6))
plt.barh(importance['feature'], importance['importance'], color='#c08fb5')
plt.title('Top 15 Features (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../docs/screenshots/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Modell speichern

In [ ]:
joblib.dump(best_model, MODELS_PATH / 'best_classifier.joblib')
print(f'Modell gespeichert: {MODELS_PATH / "best_classifier.joblib"}')

# Auch die Feature-Namen-Liste speichern (für Inferenz)
feature_meta = {
    'num_features': FEATURE_COLS_NUM,
    'cat_features': FEATURE_COLS_CAT,
    'target_classes': sorted(y.unique().tolist()),
}
import json
with open(MODELS_PATH / 'feature_meta.json', 'w') as f:
    json.dump(feature_meta, f, indent=2)
print('Feature-Meta gespeichert')

## Iterations-Tabelle (für Doku)

| Iteration | Modell | F1 (macro) Val | Accuracy Val |
| --- | --- | --- | --- |
| 1 | Logistic Regression (Baseline) | (siehe oben) | (siehe oben) |
| 2 | Random Forest | (siehe oben) | (siehe oben) |
| 3 | XGBoost | (siehe oben) | (siehe oben) |

Weiter mit `04_rag_pipeline.ipynb`.